# 4-Model Comparison: RW vs HGF × Softmax vs Race

Compares four RLSSM models on the MindRL 4-arm restless bandit:

| | Choice-only (softmax) | RT-based (race) |
|---|---|---|
| **RW** (fixed lr) | `rl_alpha`, `beta` | `rl_alpha`, `scaler`, `a`, `z`, `t`, `theta` |
| **HGF** (adaptive lr) | `omega`, `kappa`, `beta` | `omega`, `kappa`, `scaler`, `a`, `z`, `t`, `theta` |

The HGF uses adaptive learning rates via per-arm volatility tracking. The race
model adds response-time modeling via a multi-choice racing DDM.

## 1. Setup

In [ ]:
import logging, os, warnings
import arviz as az
import numpy as np
import pandas as pd
from scipy.special import logsumexp

import hssm
from ssms import rl
from ssms.rl import ModelConfig
from ssms.rl.env import Bandit

from bayesd_misfits.data import ensure_data_downloaded, load_challenge_data
from bayesd_misfits.model import NArmRescorlaWagner, NArmRWDriftLearner
from bayesd_misfits.hgf import NArmHGF, NArmHGFDriftLearner

warnings.filterwarnings("ignore")
logging.getLogger("jax._src.xla_bridge").setLevel("ERROR")
hssm.set_floatX("float32", update_jax=True)
SEED = 20260719

FULL_RUN = os.environ.get("FULL_RUN", "0") == "1"
N_SUB = 50 if FULL_RUN else 20
N_TRIALS = 120
N_CHAINS = 4 if FULL_RUN else 2
N_TUNE = 1000 if FULL_RUN else 500
N_DRAWS = 1000 if FULL_RUN else 500
print(f"participants={N_SUB} trials={N_TRIALS} chains={N_CHAINS} tune={N_TUNE} draws={N_DRAWS}")

## 2. Load data

In [ ]:
ensure_data_downloaded()
df = load_challenge_data(feedback_transform="normalize", rt_placeholder=-1.0, group_by="trajectory")

# Balanced panel
tc = df.groupby("participant_id").size()
df = df[df["participant_id"].isin(tc[tc == N_TRIALS].index)].reset_index(drop=True)
rng = np.random.default_rng(SEED)
pids = sorted(df["participant_id"].unique())
sel = rng.choice(pids, size=min(N_SUB, len(pids)), replace=False)
df = df[df["participant_id"].isin(sel)].sort_values(["participant_id", "trial_id"]).reset_index(drop=True)
pid_map = {p: i for i, p in enumerate(sorted(df["participant_id"].unique()))}
df["participant_id"] = df["participant_id"].map(pid_map)

# Choice-only data
data_choice = df[["participant_id", "trial_id", "response", "feedback"]].copy()
# RT-based data (drop missing RTs)
data_rt = df[df["rt"] > 0][["participant_id", "trial_id", "response", "rt", "feedback"]].copy()
# Re-balance RT data (may have dropped some trials)
rt_counts = data_rt.groupby("participant_id").size()
min_rt = rt_counts.min()
for pid in data_rt["participant_id"].unique():
    idx = data_rt[data_rt["participant_id"] == pid].index[:min_rt]
    # Keep only first min_rt trials per participant
data_rt = data_rt.groupby("participant_id").head(min_rt).sort_values(["participant_id", "trial_id"]).reset_index(drop=True)

print(f"Choice-only: {data_choice['participant_id'].nunique()} sub × {N_TRIALS} trials = {len(data_choice)} rows")
print(f"RT-based:    {data_rt['participant_id'].nunique()} sub × {min_rt} trials = {len(data_rt)} rows")

## 3. Build and fit all four models

In [ ]:
PE_PRIOR = {"name": "Normal", "mu": 0, "sigma": {"name": "HalfNormal", "sigma": 0.5}}

def hp(name, lo, hi, mu, sig):
    return hssm.Param(name, formula=f"{name} ~ 1 + (1|participant_id)",
        prior={"Intercept": hssm.Prior("TruncatedNormal", lower=lo, upper=hi, mu=mu, sigma=sig),
               "1|participant_id": PE_PRIOR})

env_s = Bandit.bernoulli(probabilities=[0.25]*4, response_labels=[0,1,2,3])
env_r = Bandit.gaussian(means=[0.5]*4, sds=[0.3]*4, response_labels=[0,1,2,3])

configs = {
    "RW+Softmax": ModelConfig("4AB_RW_Softmax", "RW+softmax", "inv_temp_softmax_4",
        NArmRescorlaWagner(4), env_s, response=["response"]),
    "HGF+Softmax": ModelConfig("4AB_HGF_Softmax", "HGF+softmax", "inv_temp_softmax_4",
        NArmHGF(4), env_s, response=["response"]),
    "RW+Race": ModelConfig("4AB_RW_RaceAngle", "RW+race", "race_no_bias_angle_4",
        NArmRWDriftLearner(4), env_r, response=["rt", "response"]),
    "HGF+Race": ModelConfig("4AB_HGF_RaceAngle", "HGF+race", "race_no_bias_angle_4",
        NArmHGFDriftLearner(4), env_r, response=["rt", "response"]),
}
for c in configs.values(): c.validate()

includes = {
    "RW+Softmax": [hp("rl_alpha", 0, 1, 0.2, 0.15), hp("beta", 0, 10, 3, 1.5)],
    "HGF+Softmax": [hp("omega", -8, 2, -2, 1), hp("kappa", 0, 4, 1, 0.5), hp("beta", 0, 10, 3, 1.5)],
    "RW+Race": [hp("rl_alpha", 0, 1, 0.2, 0.15), hp("scaler", 0.1, 5, 2, 0.8),
                hp("a", 0.3, 2.5, 1.2, 0.3), hp("z", 0.1, 0.9, 0.5, 0.15),
                hp("t", 0.05, 1, 0.3, 0.1), hp("theta", -0.1, 1.3, 0.3, 0.15)],
    "HGF+Race": [hp("omega", -8, 2, -2, 1), hp("kappa", 0, 4, 1, 0.5), hp("scaler", 0.1, 5, 2, 0.8),
                hp("a", 0.3, 2.5, 1.2, 0.3), hp("z", 0.1, 0.9, 0.5, 0.15),
                hp("t", 0.05, 1, 0.3, 0.1), hp("theta", -0.1, 1.3, 0.3, 0.15)],
}
data_map = {"RW+Softmax": data_choice, "HGF+Softmax": data_choice,
            "RW+Race": data_rt, "HGF+Race": data_rt}

idatas = {}
for name in configs:
    print(f"\n{'='*60}")
    print(f"Fitting {name}...")
    mc = hssm.rl.RLSSMConfig.from_ssms_model(configs[name])
    model = hssm.RLSSM(data=data_map[name], model_config=mc,
        p_outlier=0, lapse=None, process_initvals=False, include=includes[name])
    idata = model.sample(sampler="numpyro", draws=N_DRAWS, tune=N_TUNE,
        chains=N_CHAINS, cores=1, target_accept=0.9, random_seed=SEED,
        idata_kwargs={"log_likelihood": False})
    idatas[name] = idata
    div = int(idata.sample_stats["diverging"].sum())
    rh = max(float(az.rhat(idata)[v].max()) for v in az.rhat(idata).data_vars)
    print(f"  divergences={div}, max R-hat={rh:.3f}")
print("\nAll models fitted ✓")

## 4. One-step-ahead NLL comparison

In [ ]:
def draw_theta(idata, params, n_sub, idx):
    post = idata.posterior
    if hasattr(post, "to_dataset"): post = post.to_dataset()
    post = post.stack(sample=("chain", "draw"))
    theta = {}
    for name in params:
        re = post[f"{name}_1|participant_id"]
        dim = [d for d in re.dims if d != "sample"][0]
        vals = (post[f"{name}_Intercept"] + re).isel(sample=idx)
        ids = [int(v) for v in re[dim].values]
        theta[name] = pd.Series(np.asarray(vals.values), index=ids).sort_index().reindex(range(n_sub)).to_numpy()
    return theta

def compute_nll_rw(idata, data, n_sub, n_draws=100):
    """RW one-step-ahead NLL."""
    post = idata.posterior
    if hasattr(post, "to_dataset"): post = post.to_dataset()
    post = post.stack(sample=("chain", "draw"))
    n_total = post.sizes["sample"]
    didx = np.random.default_rng(42).choice(n_total, min(n_draws, n_total), replace=False)
    nlls = []
    for d in didx:
        th = draw_theta(idata, ["rl_alpha", "beta"], n_sub, int(d))
        for pid in range(n_sub):
            a, b = th["rl_alpha"][pid], th["beta"][pid]
            Q = np.full(4, 0.5)
            for _, r in data[data["participant_id"]==pid].sort_values("trial_id").iterrows():
                act, rew = int(r["response"]), float(r["feedback"])
                lg = b * Q
                nlls.append(-(lg[act] - logsumexp(lg)))
                Q[act] += a * (rew - Q[act])
    return np.array(nlls).reshape(len(didx), -1).mean(axis=1)

def compute_nll_hgf(idata, data, n_sub, n_draws=100, pi_u=20.0, th_var=0.01):
    """HGF one-step-ahead NLL."""
    post = idata.posterior
    if hasattr(post, "to_dataset"): post = post.to_dataset()
    post = post.stack(sample=("chain", "draw"))
    n_total = post.sizes["sample"]
    didx = np.random.default_rng(42).choice(n_total, min(n_draws, n_total), replace=False)
    nlls = []
    for d in didx:
        th = draw_theta(idata, ["omega", "kappa", "beta"], n_sub, int(d))
        for pid in range(n_sub):
            om, ka, be = th["omega"][pid], th["kappa"][pid], th["beta"][pid]
            mu1 = np.full(4, 0.5); s1 = np.full(4, 0.25); mu2 = np.full(4, -1.0); s2 = np.full(4, 1.0)
            for _, r in data[data["participant_id"]==pid].sort_values("trial_id").iterrows():
                act, rew = int(r["response"]), float(r["feedback"])
                mu1h = mu1.copy(); s1h = s1 + np.exp(ka*mu2+om); mu2h = mu2.copy(); s2h = s2 + th_var
                lg = be * mu1h
                nlls.append(-(lg[act] - logsumexp(lg)))
                pi1 = 1.0/s1h[act] + pi_u; psi1 = pi_u/pi1
                pe = rew - mu1h[act]
                mu1[act] = mu1h[act] + psi1 * pe; s1[act] = 1.0/pi1
                pi1h = 1.0/s1h[act]; d1 = pi1h/pi1 + pi1h*(psi1*pe)**2 - 1.0
                pi2 = 1.0/s2h[act] + 0.5*(ka*pi1h)**2; psi2 = 0.5*ka*pi1h/pi2
                mu2[act] = mu2h[act] + psi2*d1; s2[act] = 1.0/pi2
                for a2 in range(4):
                    if a2 != act: s1[a2] = s1h[a2]; s2[a2] = s2h[a2]
    return np.array(nlls).reshape(len(didx), -1).mean(axis=1)

n_sub = data_choice["participant_id"].nunique()
results = {}
results["RW+Softmax"] = compute_nll_rw(idatas["RW+Softmax"], data_choice, n_sub)
results["HGF+Softmax"] = compute_nll_hgf(idatas["HGF+Softmax"], data_choice, n_sub)

# For RT models, compute choice NLL on the same choice-only data
# (using the RT model's posterior but evaluating choice predictions only)
def compute_nll_rw_rt(idata, data, n_sub, n_draws=100):
    """RW+Race choice NLL (ignore RT, just use Q-values for softmax)."""
    post = idata.posterior
    if hasattr(post, "to_dataset"): post = post.to_dataset()
    post = post.stack(sample=("chain", "draw"))
    n_total = post.sizes["sample"]
    didx = np.random.default_rng(42).choice(n_total, min(n_draws, n_total), replace=False)
    nlls = []
    for d in didx:
        th = draw_theta(idata, ["rl_alpha", "scaler"], n_sub, int(d))
        for pid in range(n_sub):
            a, sc = th["rl_alpha"][pid], th["scaler"][pid]
            # For race model: P(choose k) ∝ v_k = scaler * Q[k]
            # This is softmax with beta=1 over v_k
            Q = np.full(4, 0.5)
            for _, r in data[data["participant_id"]==pid].sort_values("trial_id").iterrows():
                act, rew = int(r["response"]), float(r["feedback"])
                v = sc * Q  # drift rates
                nlls.append(-(v[act] - logsumexp(v)))
                Q[act] += a * (rew - Q[act])
    return np.array(nlls).reshape(len(didx), -1).mean(axis=1)

def compute_nll_hgf_rt(idata, data, n_sub, n_draws=100, pi_u=20.0, th_var=0.01):
    """HGF+Race choice NLL (ignore RT, just use belief means)."""
    post = idata.posterior
    if hasattr(post, "to_dataset"): post = post.to_dataset()
    post = post.stack(sample=("chain", "draw"))
    n_total = post.sizes["sample"]
    didx = np.random.default_rng(42).choice(n_total, min(n_draws, n_total), replace=False)
    nlls = []
    for d in didx:
        th = draw_theta(idata, ["omega", "kappa", "scaler"], n_sub, int(d))
        for pid in range(n_sub):
            om, ka, sc = th["omega"][pid], th["kappa"][pid], th["scaler"][pid]
            mu1 = np.full(4, 0.5); s1 = np.full(4, 0.25); mu2 = np.full(4, -1.0); s2 = np.full(4, 1.0)
            for _, r in data[data["participant_id"]==pid].sort_values("trial_id").iterrows():
                act, rew = int(r["response"]), float(r["feedback"])
                mu1h = mu1.copy(); s1h = s1 + np.exp(ka*mu2+om); mu2h = mu2.copy(); s2h = s2 + th_var
                v = sc * mu1h  # drift rates
                nlls.append(-(v[act] - logsumexp(v)))
                pi1 = 1.0/s1h[act] + pi_u; psi1 = pi_u/pi1
                pe = rew - mu1h[act]
                mu1[act] = mu1h[act] + psi1 * pe; s1[act] = 1.0/pi1
                pi1h = 1.0/s1h[act]; d1 = pi1h/pi1 + pi1h*(psi1*pe)**2 - 1.0
                pi2 = 1.0/s2h[act] + 0.5*(ka*pi1h)**2; psi2 = 0.5*ka*pi1h/pi2
                mu2[act] = mu2h[act] + psi2*d1; s2[act] = 1.0/pi2
                for a2 in range(4):
                    if a2 != act: s1[a2] = s1h[a2]; s2[a2] = s2h[a2]
    return np.array(nlls).reshape(len(didx), -1).mean(axis=1)

results["RW+Race"] = compute_nll_rw_rt(idatas["RW+Race"], data_choice, n_sub)
results["HGF+Race"] = compute_nll_hgf_rt(idatas["HGF+Race"], data_choice, n_sub)

print("=" * 65)
print("One-step-ahead NLL per trial (lower = better)")
print("=" * 65)
print(f"\n{'Model':<20} {'Mean':>8} {'SD':>8} {'94% HDI':>22}")
print(f"{'─'*20} {'─'*8} {'─'*8} {'─'*22}")
for name in ["RW+Softmax", "HGF+Softmax", "RW+Race", "HGF+Race"]:
    nll = results[name]
    print(f"{name:<20} {nll.mean():>8.4f} {nll.std():>8.4f} "
          f"[{np.quantile(nll, 0.03):.4f}, {np.quantile(nll, 0.97):.4f}]")
print(f"{'Uniform random':<20} {np.log(4):>8.4f}")
print()
best = min(results, key=lambda k: results[k].mean())
print(f"Best model: {best} (NLL = {results[best].mean():.4f})")

## 5. Parameter estimates

In [ ]:
for name in ["RW+Softmax", "HGF+Softmax", "RW+Race", "HGF+Race"]:
    idata = idatas[name]
    mc = hssm.rl.RLSSMConfig.from_ssms_model(configs[name])
    params = mc.list_params
    var_names = [f"{p}_Intercept" for p in params]
    print(f"\n{'='*40}")
    print(f"{name}:")
    print(az.summary(idata, var_names=var_names, kind="stats", round_to=3))

## 6. Summary

| Model | Learning | Decision | Params | NLL/trial |
|---|---|---|---|---|
| RW+Softmax | Fixed α | Softmax | 2 | see output |
| HGF+Softmax | Adaptive (volatility) | Softmax | 3 | see output |
| RW+Race | Fixed α | Race+Angle | 6 | see output |
| HGF+Race | Adaptive (volatility) | Race+Angle | 7 | see output |
| Random | — | — | — | ln(4) ≈ 1.386 |

### Key differences

- **RW vs HGF**: The HGF replaces the fixed learning rate α with adaptive
  learning rates driven by per-arm volatility estimates (ω, κ). This should
  help when different arms drift at different rates.
- **Softmax vs Race**: The softmax models choice only; the race model adds
  RT modeling (boundary separation, non-decision time, angle collapse).
  The race model's choice predictions use drift rates v_k = scaler × Q[k]
  instead of softmax with β.
- **HGF+Race**: Combines adaptive learning with RT modeling — the most
  comprehensive model, but also the most parameter-heavy.

### Why the HGF may or may not help

The HGF's advantage depends on **whether the environment has varying
volatility**. If all arms drift at roughly the same constant rate, the HGF
estimates approximately constant volatility, making it behave like a
Bayesian filter with a fixed learning rate — similar to RW. The HGF shines
when volatility changes (e.g., some arms are stable while others drift
rapidly), because it can learn fast on volatile arms and slowly on stable
ones.

### Note on convergence

With `FULL_RUN=0` (2 chains, 500 draws), expect divergences and high R-hat.
Set `FULL_RUN=1` for production-quality inference (4 chains, 1000+1000 draws).